# Leave-One-Site-Out Model Robustness

This notebook holds out each of the six lakes in turn. For each run, all pixels from the held-out lake are reserved for the final site test; whole scene IDs from the other five lakes are assigned to approximately 80% training and 20% test data. Both contain all five sites, both sensors, all seasons, and all classes. The only permitted train/test scene overlap is the configured Geneva 2021-09-06 scene, allocated as 120/30 rows whenever Geneva is not the held-out site. DT, RF, and XGBoost each receive exactly one Optuna search over one fixed stratified five-fold CV. There are no repeated CV-seed searches or candidate re-ranking passes.

In [1]:
from dataclasses import replace
import importlib
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import lswt_cloud_masking.leave_one_site_out as loso
importlib.reload(loso)
from lswt_cloud_masking.leave_one_site_out import LeaveOneSiteOutConfig, run_leave_one_site_out_pipeline

/opt/homebrew/Caskroom/miniforge/base/envs/lswt-thin-cloud/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configure the experiment

Keep `SMOKE_TEST = True` for a one-site, one-trial check. Set it to `False` for all six held-out sites with 120 trials per classifier. Completed full site runs are resumable.

In [2]:
config = LeaveOneSiteOutConfig.from_json(ROOT / "configs" / "leave_one_site_out.example.json")

SMOKE_TEST = False
if SMOKE_TEST:
    sites = config.held_out_sites or ["ageri"]
    config = replace(
        config,
        output_dir=f"{config.output_dir}_smoke",
        held_out_sites=sites[:1],
        cv_splits=2,
        n_trials_dt=1,
        n_trials_rf=1,
        n_trials_xgb=1,
        xgb_max_estimators=100,
        xgb_early_stopping_rounds=10,
        resume_completed_runs=False,
    )

print("Held-out sites:", config.held_out_sites)
print("CV folds:", config.cv_splits)
print("Trials (DT/RF/XGB):", config.n_trials_dt, config.n_trials_rf, config.n_trials_xgb)
print("Output root:", (ROOT / config.output_dir).resolve())
config

Held-out sites: ['ageri', 'bianco', 'geneva', 'greifensee', 'mendota', 'venice']
CV folds: 5
Trials (DT/RF/XGB): 200 200 200
Output root: /Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/outputs/leave_one_site_out


LeaveOneSiteOutConfig(input_csvs=['data/df_train_all.csv', 'data/df_test_all.csv'], output_dir='outputs/leave_one_site_out', held_out_sites=['ageri', 'bianco', 'geneva', 'greifensee', 'mendota', 'venice'], site_column='lakeN', scene_column='scene_id', label_column='lst_class', drop_columns=['lakeN', 'scene_id', 'lst_raw', 'lst_filt', 'raamap', 'vzamap', 'szamap'], test_fraction=0.2, split_random_state=42, scene_split_search_iterations=100000, scene_split_max_test_fraction_deviation=0.01, overlap_exception_scene_id='LC08_L1TP_196028_20210906_20210915_02_T1', overlap_exception_site='geneva', overlap_exception_class=3, overlap_exception_train_rows=120, overlap_exception_test_rows=30, cv_splits=5, cv_random_state=42, model_random_state=42, optuna_seed=42, scoring='balanced_accuracy', n_trials_dt=200, n_trials_rf=200, n_trials_xgb=200, optuna_n_jobs=1, rf_n_jobs=-1, xgb_n_jobs=-1, xgb_sample_weight='balanced', xgb_max_estimators=1500, xgb_early_stopping_rounds=250, resume_completed_runs=Tru

## 2. Create splits, tune once, refit, and evaluate

The approximately 20% scene-disjoint test and held-out lake are never passed to Optuna or XGBoost early stopping. XGBoost early stopping uses only each training fold's internal validation fold.

In [3]:
loso_result = run_leave_one_site_out_pipeline(config, project_root=ROOT)
loso_result["paths"]

[1/6] Creating 80/20 split with 'ageri' completely held out.


[I 2026-08-25 14:38:13,264] A new study created in memory with name: DecisionTree_LOSO


[1/6] Running one 5-fold search per model.
  ageri: tuning Decision Tree (200 trials).


[I 2026-08-25 14:38:13,810] Trial 0 finished with value: 0.6595973227540484 and parameters: {'max_depth': 12, 'min_samples_split': 96, 'min_samples_leaf': 59, 'max_features': 0.6, 'criterion': 'log_loss', 'splitter': 'best', 'class_weight': 'balanced', 'ccp_alpha': 1.3480180290890803e-07}. Best is trial 0 with value: 0.6595973227540484.
[I 2026-08-25 14:38:13,876] Trial 1 finished with value: 0.5398143487544739 and parameters: {'max_depth': 17, 'min_samples_split': 44, 'min_samples_leaf': 24, 'max_features': 0.8, 'criterion': 'log_loss', 'splitter': 'random', 'class_weight': None, 'ccp_alpha': 0.0043873144324354085}. Best is trial 0 with value: 0.6595973227540484.
[I 2026-08-25 14:38:13,904] Trial 2 finished with value: 0.5070742159351423 and parameters: {'max_depth': 30, 'min_samples_split': 82, 'min_samples_leaf': 25, 'max_features': 'sqrt', 'criterion': 'gini', 'splitter': 'random', 'class_weight': None, 'ccp_alpha': 0.006124806805925988}. Best is trial 0 with value: 0.6595973227540

  ageri: tuning Random Forest (200 trials).


...

[I 2026-08-27 05:41:56,478] Trial 0 finished with value: 0.8142192395082845 and parameters: {'n_estimators': 300, 'max_depth': 32, 'min_samples_split': 3, 'min_samples_leaf': 15, 'max_features': 0.7, 'bootstrap': True, 'criterion': 'log_loss', 'class_weight': None}. Best is trial 0 with value: 0.8142192395082845.
[I 2026-08-27 05:42:06,657] Trial 1 finished with value: 0.7841686774269815 and parameters: {'n_estimators': 600, 'max_depth': 16, 'min_samples_split': 20, 'min_samples_leaf': 13, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'gini', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8142192395082845.
[I 2026-08-27 05:42:38,756] Trial 2 finished with value: 0.6873521315223428 and parameters: {'n_estimators': 600, 'max_depth': 8, 'min_samples_split': 28, 'min_samples_leaf': 15, 'max_features': 0.7, 'bootstrap': True, 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8142192395082845.
[I 2026-08-27 05:43:04,752] Trial 3 finished

  venice: tuning XGBoost (200 trials).


[I 2026-08-27 08:19:53,600] Trial 0 finished with value: 0.8330644788546315 and parameters: {'max_depth': 9, 'learning_rate': 0.019677977348166304, 'subsample': 0.8978561926718072, 'colsample_bytree': 0.7442003345025142, 'min_child_weight': 2.1771032186203514, 'gamma': 1.8277151418544553, 'reg_alpha': 8.665537597324885e-06, 'reg_lambda': 1.0311514495138399}. Best is trial 0 with value: 0.8330644788546315.
[I 2026-08-27 08:20:09,857] Trial 1 finished with value: 0.8554768025671576 and parameters: {'max_depth': 7, 'learning_rate': 0.09439831569936806, 'subsample': 0.9842104900822669, 'colsample_bytree': 0.7826484435755748, 'min_child_weight': 2.880043790954131, 'gamma': 0.3403911033599112, 'reg_alpha': 1.5100408408589772e-07, 'reg_lambda': 23.538263110950048}. Best is trial 1 with value: 0.8554768025671576.
[I 2026-08-27 08:20:25,839] Trial 2 finished with value: 0.8727881912840747 and parameters: {'max_depth': 10, 'learning_rate': 0.14726161255504092, 'subsample': 0.8585642249613574, 'c

LOSO comparison written to /Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/outputs/leave_one_site_out/leave_one_site_out_results.csv.


{'results': '/Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/outputs/leave_one_site_out/leave_one_site_out_results.csv',
 'summary': '/Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/outputs/leave_one_site_out/leave_one_site_out_summary_by_model.csv',
 'manifest': '/Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/outputs/leave_one_site_out/leave_one_site_out_manifest.json'}

## 3. Inspect per-site and across-site robustness

In [4]:
columns = [
    "held_out_site", "model",
    "train_balanced_accuracy",
    "test_balanced_accuracy",
    "leave_out_site_balanced_accuracy",
    "cv_selection_score_mean",
    "cv_selection_score_std",
    "model_path",
]
loso_result["results"][columns]

,held_out_site,model,train_balanced_accuracy,test_balanced_accuracy,leave_out_site_balanced_accuracy,cv_selection_score_mean,cv_selection_score_std,model_path
0,ageri,decision_tree,0.979093,0.514771,0.447309,0.770884,0.005259,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
1,ageri,random_forest,1.000000,0.591702,0.447770,0.866919,0.004889,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
2,ageri,xgboost,1.000000,0.589719,0.450999,0.887538,0.006792,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
3,bianco,decision_tree,0.998832,0.513764,0.223253,0.783450,0.007787,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
4,bianco,random_forest,1.000000,0.556266,0.172798,0.876120,0.004257,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
5,bianco,xgboost,1.000000,0.561460,0.395196,0.893878,0.003507,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
6,geneva,decision_tree,1.000000,0.475172,0.462393,0.829066,0.006319,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
7,geneva,random_forest,1.000000,0.511905,0.488964,0.903275,0.005840,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
8,geneva,xgboost,1.000000,0.517801,0.496206,0.919213,0.008527,/Users/airahaghi/Documents/Trishna/Scripts/LSW...
9,greifensee,decision_tree,0.993749,0.465735,0.536441,0.785863,0.003106,/Users/airahaghi/Documents/Trishna/Scripts/LSW...


In [5]:
loso_result["summary"]

,model,n_sites,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_balanced_accuracy_min,train_balanced_accuracy_max,test_balanced_accuracy_mean,test_balanced_accuracy_std,test_balanced_accuracy_min,test_balanced_accuracy_max,leave_out_site_balanced_accuracy_mean,leave_out_site_balanced_accuracy_std,leave_out_site_balanced_accuracy_min,leave_out_site_balanced_accuracy_max
0,decision_tree,6,0.982964,0.022264,0.940556,1.0,0.496778,0.022304,0.465735,0.517624,0.436049,0.108866,0.223253,0.536441
1,random_forest,6,1.000000,0.000000,1.000000,1.0,0.546315,0.030895,0.510546,0.591702,0.471952,0.164126,0.172798,0.665474
2,xgboost,6,1.000000,0.000000,1.000000,1.0,0.549251,0.027946,0.517088,0.589719,0.519141,0.100344,0.395196,0.687257


Each `datasets/held_out_*` folder contains its train, scene-disjoint test, held-out-site CSVs, split summary, and `scene_split_manifest.csv`. Each parallel `models/held_out_*` folder contains the three final models, one study and trials table per classifier, detailed metrics, confusion matrices, classification reports, and metadata. The root results CSV provides the 18 site/model comparisons.